In [13]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments, Trainer
import sys
from pathlib import Path
from datasets import Dataset, DatasetDict
import torch
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

NOTEBOOK_ROOT = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_ROOT.parent if NOTEBOOK_ROOT.name == "src" else NOTEBOOK_ROOT
SRC_ROOT = PROJECT_ROOT / "src"

for import_root in (PROJECT_ROOT, SRC_ROOT):
    if str(import_root) not in sys.path:
        sys.path.append(str(import_root))

from dataset_loader import load_prolog_samples
from data.pipeline_constants import MAX_TRAIN_SAMPLE_TOKENS
import os

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
model_dtype = torch.bfloat16 if use_bf16 else torch.float16

In [14]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [15]:
model_id = os.getenv('TRAIN_BASE_MODEL_ID')
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=model_dtype,
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    torch_dtype=model_dtype,
    device_map="auto",
    trust_remote_code=True,
)
model.config.pad_token_id = tokenizer.pad_token_id
model.generation_config.pad_token_id = tokenizer.pad_token_id
old_base_test_messages = [
    {"role": "system", "content": "Ты пишешь только код на Prolog."},
    {"role": "user", "content": "Реализуй предикат member_of(Element, List), который истинен, если элемент входит в список."},
]
base_test_messages = [
    {"role": "system", "content": "Ты пишешь только код на SWI-Prolog без пояснений."},
    {
        "role": "user",
        "content": "Реализуй предикат last_element(List, Element), который истинен, если Element — последний элемент списка. Верни только код.",
    },
]
model.device

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
W0530 23:41:53.854000 4412 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
Loading weights:   0%|          | 1/290 [00:00<02:03,  2.33it/s]c:\Projects\TgSummarizer\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 290/290 [00:01<00:00, 236.37it/s]


device(type='cuda', index=0)

In [16]:
inputs = tokenizer.apply_chat_template(
    base_test_messages,
    tokenize=True,
    enable_thinking=False,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

# Генерируем ответ
generated_ids = model.generate(**inputs, max_new_tokens=160)
new_tokens = generated_ids[:, inputs["input_ids"].shape[1]:]
print(tokenizer.batch_decode(new_tokens, skip_special_tokens=True))

c:\Projects\TgSummarizer\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


['Code in Prolog Africa\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nForgery\nFor

In [42]:
print(model.device)

cuda:0


In [43]:
tokenizer.eos_token_id

151643

In [44]:
tokenizer.pad_token_id

151643

In [45]:
def load_prolog_data_legacy(data_dir):
    """
    Загружает .pl файлы и соответствующие .txt описания в формате JSON.
    Структура: data_dir/annotated_repos/repo_name/*.pl и *.txt
    """
    data = []
    repos_path = Path(data_dir) / "annotated_repos"
    
    if not repos_path.exists():
        raise ValueError(f"Папка {repos_path} не найдена")
    
    for repo_dir in repos_path.iterdir():
        if not repo_dir.is_dir():
            continue
            
        print(f"📁 Обрабатываю репозиторий: {repo_dir.name}")
        
        # Проходим по всем .pl файлам в репозитории
        for pl_file in repo_dir.glob('*.pl'):
            txt_file = pl_file.with_suffix('.txt')
            
            # Проверяем наличие .txt файла
            if not txt_file.exists():
                print(f"  ⚠️  Нет описания для {pl_file.name} — пропускаю")
                continue
            
            description = txt_file.read_text(encoding='utf-8').strip()
                
            try:
                code = pl_file.read_text(encoding='utf-8').strip()
            except Exception as e:
                print(f"  ❌ Ошибка чтения {pl_file.name}: {e}")
                continue
            
            messages = [
                        {
                            "role": "system",
                            "content": (
                            "Вы — экспертный разработчик на Prolog. "
                            "Генерируйте чистый, идиоматичный код на Prolog на основе описания. "
        
                        )
                        },
                        {
                            "role": "user",
                            "content": description
                        },
                        {
                            "role": "assistant",
                            "content": f"\n{code}\n"
                        }
                    ]
            
            data.append(messages)
            
            
    
    print(f"\n📊 Всего загружено примеров: {len(data)}")
    return data


def load_dataset(data_dir):
    samples = load_prolog_samples(
        data_dir,
        max_train_sample_tokens=MAX_TRAIN_SAMPLE_TOKENS,
        recalc_token_counts=True,
    )
    print(f"Loaded samples: {len(samples)}")
    return samples

In [46]:
# === Использование ===

train_samples = load_dataset('../data')




📁 Обрабатываю репозиторий: 2kodevs_Azul-Game
📁 Обрабатываю репозиторий: acharal_yadlr
📁 Обрабатываю репозиторий: ai-unibo_log-generator
📁 Обрабатываю репозиторий: amarusofi_LMC
📁 Обрабатываю репозиторий: amka66_mai
📁 Обрабатываю репозиторий: AndreaInfUFSM_elc117-2022a
📁 Обрабатываю репозиторий: andyleejordan_uidaho-cs470-prolog
📁 Обрабатываю репозиторий: anightatheopera_fichasIA
📁 Обрабатываю репозиторий: arnavk_prolog
📁 Обрабатываю репозиторий: Baha_maper
📁 Обрабатываю репозиторий: blueridanus_prologquest
📁 Обрабатываю репозиторий: BoseSean_CZ3005
📁 Обрабатываю репозиторий: brunocampos01_prolog-language
📁 Обрабатываю репозиторий: bsspirit_prolog-learning
📁 Обрабатываю репозиторий: byvlstr_DotsAndBoxesProlog
📁 Обрабатываю репозиторий: cbarrick_normalization
📁 Обрабатываю репозиторий: cchrewrite_SLDR-DL
📁 Обрабатываю репозиторий: COMS30106_prolog_intro
📁 Обрабатываю репозиторий: dg1an3_ALGT
📁 Обрабатываю репозиторий: diveshuttam_CS-F214
📁 Обрабатываю репозиторий: divyaparadkar_Prolog.pl

In [47]:
print(len(train_samples))

769


In [48]:
def render_full_text(messages):
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        enable_thinking=False,
        add_generation_prompt=False,
    )

def tokenize_messages(messages):
    full_text = render_full_text(messages)
    full_ids = tokenizer(
        full_text,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_TRAIN_SAMPLE_TOKENS,
    )["input_ids"]

    prefix_text = tokenizer.apply_chat_template(
        messages[:-1],
        tokenize=False,
        enable_thinking=False,
        add_generation_prompt=True,
    )
    prefix_ids = tokenizer(
        prefix_text,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_TRAIN_SAMPLE_TOKENS,
    )["input_ids"]

    prefix_len = min(len(prefix_ids), len(full_ids))
    labels = [-100] * prefix_len + full_ids[prefix_len:]

    return {
        "input_ids": full_ids,
        "attention_mask": [1] * len(full_ids),
        "labels": labels,
    }

train_texts = [render_full_text(sample.messages) for sample in train_samples]
dataset_texts = train_texts

dataset = DatasetDict({
    "train": Dataset.from_list([{"messages": sample.messages} for sample in train_samples]),
})

In [49]:
import numpy as np
lengths = [len(tokenizer.encode(txt)) for txt in dataset_texts]
np.percentile(lengths, 95)

np.float64(5983.999999999992)

In [ ]:
def tokenize_func(examples):
    rows = [tokenize_messages(messages) for messages in examples["messages"]]
    return {
        "input_ids": [row["input_ids"] for row in rows],
        "attention_mask": [row["attention_mask"] for row in rows],
        "labels": [row["labels"] for row in rows],
    }

dataset = dataset.map(tokenize_func, batched=True, remove_columns=["messages"])
print(f"Train: {len(dataset['train'])}")

Map: 100%|██████████| 769/769 [00:01<00:00, 430.60 examples/s]

Train: 692, Val: 77


In [51]:
print(tokenizer.decode(dataset['train'][50]["input_ids"]))

<|im_start|>system
Вы — экспертный разработчик на Prolog. Генерируйте чистый, идиоматичный код на Prolog на основе описания. <|im_end|>
<|im_start|>user
Программа реализует систему логических связей между членами семьи позволяя определять различные родственные отношения на основе базовых фактов о родительских связях и поле. Основной входной информацией служат данные о том кто является родителем кого а также информация о поле каждого члена семьи. Система строит логические выводы позволяя устанавливать связи между людьми не указанными напрямую в базе фактов но связанными через цепочку родительских отношений. Система позволяет определять такие отношения как мать отец дочь сын брат сестра дедушка бабушка тетя дядя племянник племянница предок и прародитель. В процессе работы предикаты строят цепочки логических связей используя базовые отношения между детьми и родителями а также устанавливают дополнительные связи на основе брака и родственных связей. Например если известно что один человек я

In [52]:
OUTPUT_DIR = "./prolog_model"           # Куда сохранить результат
EPOCHS = 3                             # Сколько раз прогнать данные
BATCH_SIZE = 1   

In [53]:
for name, _ in model.named_modules():
    print(name)


model
model.embed_tokens
model.layers
model.layers.0
model.layers.0.self_attn
model.layers.0.self_attn.q_proj
model.layers.0.self_attn.k_proj
model.layers.0.self_attn.v_proj
model.layers.0.self_attn.o_proj
model.layers.0.mlp
model.layers.0.mlp.gate_proj
model.layers.0.mlp.up_proj
model.layers.0.mlp.down_proj
model.layers.0.mlp.act_fn
model.layers.0.input_layernorm
model.layers.0.post_attention_layernorm
model.layers.1
model.layers.1.self_attn
model.layers.1.self_attn.q_proj
model.layers.1.self_attn.k_proj
model.layers.1.self_attn.v_proj
model.layers.1.self_attn.o_proj
model.layers.1.mlp
model.layers.1.mlp.gate_proj
model.layers.1.mlp.up_proj
model.layers.1.mlp.down_proj
model.layers.1.mlp.act_fn
model.layers.1.input_layernorm
model.layers.1.post_attention_layernorm
model.layers.2
model.layers.2.self_attn
model.layers.2.self_attn.q_proj
model.layers.2.self_attn.k_proj
model.layers.2.self_attn.v_proj
model.layers.2.self_attn.o_proj
model.layers.2.mlp
model.layers.2.mlp.gate_proj
model.l

In [54]:
target_modules = ["q_proj", "k_proj", "v_proj"]


In [55]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=target_modules,
    task_type="CAUSAL_LM",
    
)
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    learning_rate=2e-5,
    logging_steps=50,
    save_strategy="epoch",
    bf16=use_bf16,
    fp16=not use_bf16,
    report_to="none"          # Отключить лишние логи
)


In [56]:
def sft_collator(features):
    batch = tokenizer.pad(
        [
            {
                "input_ids": feature["input_ids"],
                "attention_mask": feature["attention_mask"],
            }
            for feature in features
        ],
        padding=True,
        return_tensors="pt",
    )
    max_length = batch["input_ids"].shape[1]
    labels = [
        feature["labels"] + [-100] * (max_length - len(feature["labels"]))
        for feature in features
    ]
    batch["labels"] = torch.tensor(labels, dtype=torch.long)
    return batch

In [57]:
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model,lora_config)
model.config.use_cache = False

In [58]:
model.print_trainable_parameters()

trainable params: 737,280 || all params: 494,770,048 || trainable%: 0.1490


In [59]:
print(dataset["train"][0].keys())

dict_keys(['input_ids', 'attention_mask', 'labels'])


In [60]:
model.gradient_checkpointing_enable()  

In [65]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

In [62]:
trainer = Trainer(model,args=training_args,data_collator=sft_collator,train_dataset=dataset['train'])

In [63]:
trainer.train()

Step,Training Loss
50,1.606824
100,1.559565
150,1.588394
200,1.472179
250,1.500914
300,1.428421
350,1.481026
400,1.486797
450,1.425021
500,1.391564


TrainOutput(global_step=2076, training_loss=1.385623590105531, metrics={'train_runtime': 1513.4739, 'train_samples_per_second': 1.372, 'train_steps_per_second': 1.372, 'total_flos': 2.233581221376e+16, 'train_loss': 1.385623590105531, 'epoch': 3.0})

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import os

base_model_id = os.getenv('TRAIN_BASE_MODEL_ID')
adapter_path = "./prolog_model/checkpoint-1384"  # или "./prolog_model", если там уже лежит финальная модель

print("Загружаю базовую модель и токенизатор...")
tokenizer = AutoTokenizer.from_pretrained(base_model_id, trust_remote_code=True)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id, torch_dtype="auto", device_map="cpu", trust_remote_code=True
)

print("Загружаю и объединяю LoRA-адаптер...")
model = PeftModel.from_pretrained(base_model, adapter_path)
model = model.merge_and_unload()

merged_path = "./prolog_merged"
print(f"Сохраняю полную модель в {merged_path}...")
model.save_pretrained(merged_path)
tokenizer.save_pretrained(merged_path)
print("Объединение завершено!")

Загружаю базовую модель и токенизатор...


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 8606.20it/s]


Загружаю и объединяю LoRA-адаптер...
Сохраняю полную модель в ./prolog_merged...


Writing model shards: 100%|██████████| 1/1 [00:12<00:00, 12.15s/it]


Объединение завершено!
